# Energy Transformers: ListOps Model Combinations on Cloud GPUs

This notebook runs `listops_model_combinations.py` on Google Colab or Kaggle Notebooks with automatic platform detection.

**Prerequisites:**
- W&B API key (you'll be prompted to login)
- WANDB_ENTITY environment variable set before running the script
- GPU runtime enabled (set in notebook settings)

## 1. Platform Detection & Setup

In [ ]:
import os
import sys
from pathlib import Path

# Platform detection
def detect_platform():
    """Detect if running on Colab, Kaggle, or local."""
    
    if os.path.exists('/kaggle'):
        return 'kaggle'
    
    try:
        # Note: kaggle also supports google.colab!  
        # Thats why we check for kaggle first, then colab.
        import google.colab
        return 'colab'
    except ImportError:
        pass
    return 'local'

platform = detect_platform()
print(f"🖥️  Detected platform: {platform.upper()}")

# Setup paths based on platform
if platform == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_dir = '/content/energy-transformers-tcrp'
    output_base = '/content/drive/MyDrive/energy-transformers-experiments'
elif platform == 'kaggle':
    repo_dir = '/kaggle/working/energy-transformers-tcrp'
    output_base = '/kaggle/working/outputs'
else:
    repo_dir = os.getcwd()  # Assume already in repo
    output_base = os.path.join(repo_dir, 'outputs')

print(f"📂 Repo directory: {repo_dir}")
print(f"💾 Output base: {output_base}")

🖥️  Detected platform: LOCAL
📂 Repo directory: /home/luisegzb/Development/energy-transformers-tcrp/notebooks
💾 Output base: /home/luisegzb/Development/energy-transformers-tcrp/notebooks/outputs


## 2. Clone Repository (if needed)

In [2]:
if platform != 'local':
    if not os.path.exists(repo_dir):
        print(f"📥 Cloning repository to {repo_dir}...")
        os.system(f"git clone https://github.com/luisemilio14/energy-transformers-tcrp.git {repo_dir}")
    else:
        print(f"✅ Repository already exists at {repo_dir}")
else:
    print("ℹ️  Running locally, assuming repo is current directory.")

ℹ️  Running locally, assuming repo is current directory.


## 3. Navigate to Repo & Install Dependencies

In [ ]:
os.chdir(repo_dir)
print(f"📍 Current directory: {os.getcwd()}")

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install poetry


📍 Current directory: /home/luisegzb/Development/energy-transformers-tcrp/notebooks

📦 Installing dependencies...


2

In [ ]:
!poetry install

## 4. Authenticate with Weights & Biases

In [4]:
import wandb

print("🔐 Authenticating with Weights & Biases...")
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


🔐 Authenticating with Weights & Biases...


wandb: Currently logged in as: luisegzb (luisegzb-universidade-federal-de-minas-gerais) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 5. Set Environment Variables

In [5]:
# Prompt user for WANDB_ENTITY
wandb_entity = input("🏷️  Enter your WANDB_ENTITY (username or team name): ").strip()

if not wandb_entity:
    raise ValueError("WANDB_ENTITY cannot be empty!")

os.environ['WANDB_ENTITY'] = wandb_entity
print(f"✅ Set WANDB_ENTITY={wandb_entity}")

ValueError: WANDB_ENTITY cannot be empty!

## 6. Prepare Output Directory

In [6]:
# Create output directory
Path(output_base).mkdir(parents=True, exist_ok=True)
print(f"✅ Output directory ready: {output_base}")

# For local/Kaggle, also set an environment variable for the script if needed
os.environ['OUTPUT_DIR'] = output_base

✅ Output directory ready: /home/luisegzb/Development/energy-transformers-tcrp/notebooks/outputs


## 7. GPU Detection & Setup

In [ ]:
import torch

# Detect available GPUs
n_gpus = torch.cuda.device_count()
print(f"🖥️  Detected {n_gpus} GPU(s):")

for i in range(n_gpus):
    gpu_name = torch.cuda.get_device_name(i)
    gpu_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"   GPU {i}: {gpu_name} ({gpu_memory:.1f}GB)")

if n_gpus == 0:
    raise RuntimeError("❌ No GPUs detected! This notebook requires GPU acceleration.")
elif n_gpus == 1:
    print("⚠️  Only 1 GPU detected. Parallel execution will use single GPU.")
else:
    print(f"✅ Running {n_gpus} parallel sweep agents (one per GPU)")

## 8. Run Parallel Sweep Agents on Each GPU

In [ ]:
!poetry run dvc repro listops32_tokenize

In [ ]:
import subprocess
import threading
import time
from queue import Queue

print("\n" + "="*70)
print(f"🚀 Launching {n_gpus} parallel sweep agents...")
print("="*70)

script_path = os.path.join(repo_dir, 'src', 'listops_model_combinations.py')
if not os.path.exists(script_path):
    raise FileNotFoundError(f"Script not found at {script_path}")

sys.path.insert(0, os.path.join(repo_dir, 'src'))

# Queue to track results from each GPU
results_queue = Queue()
threads = []

def run_agent_on_gpu(gpu_id, gpu_count):
    """Run sweep agent on a specific GPU."""
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    
    print(f"\n🔧 [GPU {gpu_id}] Setting up sweep agent...")
    print(f"   CUDA_VISIBLE_DEVICES={gpu_id}")
    
    try:
        start_time = time.time()
        
        result = subprocess.run(
            [sys.executable, script_path, '--config=params.yaml'],
            cwd=repo_dir,
            env=env,
            capture_output=False,
            text=True
        )
        
        elapsed = time.time() - start_time
        
        if result.returncode == 0:
            status = "✅ COMPLETED"
            print(f"\n{status} [GPU {gpu_id}] Sweep agent finished in {elapsed:.1f}s")
        else:
            status = "❌ FAILED"
            print(f"\n{status} [GPU {gpu_id}] Exit code: {result.returncode}")
            if result.stderr:
                print(f"   Error output:\n{result.stderr[:500]}")
        
        results_queue.put({
            'gpu_id': gpu_id,
            'status': 'success' if result.returncode == 0 else 'failed',
            'elapsed': elapsed,
            'returncode': result.returncode
        })
        
    except Exception as e:
        print(f"\n❌ [GPU {gpu_id}] Exception: {e}")
        results_queue.put({
            'gpu_id': gpu_id,
            'status': 'error',
            'elapsed': time.time() - start_time,
            'error': str(e)
        })

# Launch one thread per GPU
for gpu_id in range(n_gpus):
    thread = threading.Thread(target=run_agent_on_gpu, args=(gpu_id, n_gpus), daemon=False)
    thread.start()
    threads.append(thread)
    print(f"\n🧵 Thread {gpu_id} started for GPU {gpu_id}")

# Wait for all threads to complete
print(f"\n⏳ Waiting for {n_gpus} GPU agent(s) to complete...")
for thread in threads:
    thread.join()

# Collect results
results = []
while not results_queue.empty():
    results.append(results_queue.get())

# Sort by GPU ID for display
results.sort(key=lambda x: x['gpu_id'])

print("\n" + "="*70)
print("📊 Parallel Execution Summary:")
print("="*70)
for r in results:
    status_icon = "✅" if r['status'] == 'success' else "❌"
    print(f"{status_icon} GPU {r['gpu_id']}: {r['status'].upper()} ({r['elapsed']:.1f}s)")

# Check if all succeeded
all_succeeded = all(r['status'] == 'success' for r in results)
if all_succeeded:
    print(f"\n✅ All {n_gpus} sweep agent(s) completed successfully!")
else:
    failed_gpus = [r['gpu_id'] for r in results if r['status'] != 'success']
    print(f"\n⚠️  Some agents failed on GPUs: {failed_gpus}")

## 9. Post-Execution Summary

In [ ]:
print("\n📋 Execution Summary:")
print(f"  Platform: {platform.upper()}")
print(f"  GPUs Used: {n_gpus}")
print(f"  W&B Entity: {os.environ.get('WANDB_ENTITY', 'NOT SET')}")
print(f"  Output Directory: {output_base}")

# List output files if any
if os.path.exists(output_base):
    files = list(Path(output_base).glob('**/*'))[:10]  # First 10 items
    if files:
        print(f"\n  Output files ({len(files)} shown):")
        for f in files:
            if f.is_file():
                print(f"    - {f.relative_to(output_base)}")

if platform == 'colab':
    print(f"\n💾 Results are saved to Google Drive: /MyDrive/energy-transformers-experiments")
    print("   Access them via Files panel on the left.")
elif platform == 'kaggle':
    print(f"\n💾 Results are saved to: /kaggle/working/outputs")
    print("   Download them from the Output section after notebook completes.")
else:
    print(f"\n💾 Results are saved locally to: {output_base}")

print(f"\n🎯 Next Steps:")
print(f"   - Check W&B dashboard for {n_gpus} parallel runs")
print(f"   - Merge results from all GPUs for final analysis")